# Phase 5 — 01 A1 Full

Legal Statute Identification (multi-label). Run from top to bottom on Kaggle.

The notebook writes the Python modules, trains using the public Phase-5 train split, scores dev predictions with the **supplied course evaluator**, and makes a Codabench ZIP.

It needs the original Phase-1 checkpoint and tokenizer. If a repository is private, set `HF_TOKEN` on Kaggle Secrets. Results are not precomputed.


**Kaggle:** turn on a GPU and Internet in notebook settings. Add a Kaggle Secret named `HF` with a Hugging Face write token, and grant this notebook access to the secret.


In [1]:
# Install packages used in this notebook.
%pip -q install "transformers>=4.45,<5" tokenizers "huggingface_hub>=0.28" safetensors scikit-learn joblib hf_xet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Kaggle paths and HF login.
import os, sys, json, subprocess
from pathlib import Path
from huggingface_hub import HfApi
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Give each experiment its own folder in /kaggle/working.
WORKDIR = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()) / 'phase5_a1_full'
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# If Phase 5 was added as a Kaggle Input dataset, use that copy.
# Otherwise the scripts download the official Phase 5 data from Hugging Face.
DATA_DIR = ''
input_root = Path('/kaggle/input')
if not DATA_DIR and input_root.is_dir():
    for train_path in input_root.rglob('Phase_5_LSI/Train/train.json'):
        possible_root = train_path.parents[2]
        if (possible_root / 'Phase_5_LSI/Test/test.json').is_file():
            DATA_DIR = str(possible_root)
            break

# Leave these empty to use the Phase 1 repos saved in phase5_core.py.
PHASE1_SOURCE = ''
PHASE1_WEIGHT = ''

# Read HF token from a Kaggle Secret named HF (or HF_TOKEN).
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ('HF', 'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN'):
            try:
                HF_TOKEN = secrets.get_secret(key)
            except Exception:
                pass
            if HF_TOKEN:
                break
    except ImportError:
        pass
if not HF_TOKEN:
    raise RuntimeError('Add HF (write-access token) in Kaggle Add-ons > Secrets, and enable it for this notebook.')
os.environ['HF_TOKEN'] = HF_TOKEN  # Used by checkpoint/data downloads. Do not print the token.

hf_api = HfApi(token=HF_TOKEN)
HF_USERNAME = hf_api.whoami()['name']  # Get the account name from the token.
REPO_NAME = 'cs781-phase5-a1-full'
REPO_ID = f'{HF_USERNAME}/{REPO_NAME}'

# A Phase 1 backbone is too large for practical CPU training.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in Kaggle Notebook Settings before training.')
print('Using GPU:', torch.cuda.get_device_name(0))

EPOCHS = 5
BATCH_SIZE = 1
ACCUM = 4
LENGTH = 256
CHUNKS = 3
SEED = 44
print('Working folder:', WORKDIR)
print('Dataset:', DATA_DIR or 'official HF dataset')
print('HF account:', HF_USERNAME)
print('Output repo:', REPO_ID)


Using GPU: Tesla T4
Working folder: /kaggle/working/phase5_a1_full
Dataset: official HF dataset
HF account: P-261110044
Output repo: P-261110044/cs781-phase5-a1-full


## Source files

Run the following cells to write the model and evaluation scripts locally.


In [3]:
%%writefile phase5_core.py
"""Models, dataset loading and prediction helpers for Phase 5.

The Phase-1 model definitions below are copied from the earlier notebooks so the
pretrained state dictionaries still load. Phase-5 code starts near DATA_REPO.
"""
from __future__ import annotations
import os, re, json, math, random, copy, warnings
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
# Optional at import time: validator can run without Hugging Face transformers.
try:
    from transformers import PreTrainedTokenizerFast
    from transformers.modeling_outputs import CausalLMOutput
except ImportError:
    PreTrainedTokenizerFast = None
    from types import SimpleNamespace
    def CausalLMOutput(loss=None,logits=None): return SimpleNamespace(loss=loss,logits=logits)
from huggingface_hub import HfApi, hf_hub_download

# Phase-1 models used in the previous runs (kept unchanged)

# Original notebook: a1_full.ipynb
class DecoderBlock(nn.Module):
    """
    One Transformer decoder block.

    Structure:
        LayerNorm
        -> Causal Multi-Head Self Attention
        -> Residual Connection
        -> LayerNorm
        -> Feed Forward Network
        -> Residual Connection
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Normalize before self-attention.
        self.norm1 = nn.LayerNorm(
            d_model
        )

        # Multi-head self-attention.
        #
        # Query, Key and Value all come from
        # the same token sequence.
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Normalize before feed-forward network.
        self.norm2 = nn.LayerNorm(
            d_model
        )

        # Transformer feed-forward network.
        self.feed_forward = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        causal_mask
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        z = self.norm1(x)

        attention_output, _ = self.attention(
            z,
            z,
            z,
            attn_mask=causal_mask,
            need_weights=False
        )

        # Residual connection.
        x = x + self.dropout(
            attention_output
        )

        # ----------------------------------------------------
        # 2. Feed-forward network
        # ----------------------------------------------------

        z = self.norm2(x)

        ff_output = self.feed_forward(
            z
        )

        # Second residual connection.
        x = x + self.dropout(
            ff_output
        )

        return x

class DecoderTransformerLM(nn.Module):
    """
    Complete custom decoder-only Transformer
    language model.
    """

    def __init__(
        self,
        vocab_size,
        context_length=2048,
        d_model=640,
        n_heads=10,
        n_layers=8,
        d_ff=2560,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = (
            context_length
        )

        self.vocab_size = (
            vocab_size
        )

        # Safety check:
        # embedding dimension must divide evenly
        # across attention heads.
        assert (
            d_model % n_heads == 0
        ), (
            "d_model must be divisible "
            "by n_heads"
        )

        # ----------------------------------------------------
        # Token embeddings
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # ----------------------------------------------------
        # Learned positional embeddings
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            context_length,
            d_model
        )

        # ----------------------------------------------------
        # Stack of custom Transformer decoder blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization.
        self.final_norm = nn.LayerNorm(
            d_model
        )

        # Convert hidden representation
        # into vocabulary logits.
        self.output_layer = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )


    def forward(
        self,
        input_ids,
        labels=None,
        attention_mask=None,
        **kwargs
    ):

        _, seq_len = input_ids.shape

        if seq_len > self.context_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"context length {self.context_length}"
            )

        # ----------------------------------------------------
        # 1. Token embeddings
        # ----------------------------------------------------

        token_vectors = (
            self.token_embedding(
                input_ids
            )
        )

        # ----------------------------------------------------
        # 2. Positional embeddings
        # ----------------------------------------------------

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        position_vectors = (
            self.position_embedding(
                positions
            )
        )

        x = (
            token_vectors
            +
            position_vectors
        )

        # ----------------------------------------------------
        # 3. Causal mask
        #
        # True = attention blocked.
        #
        # Token i cannot see tokens after i.
        # ----------------------------------------------------

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # 4. Transformer blocks
        # ----------------------------------------------------

        for block in self.blocks:

            x = block(
                x,
                causal_mask
            )

        # ----------------------------------------------------
        # 5. Vocabulary prediction
        # ----------------------------------------------------

        x = self.final_norm(
            x
        )

        logits = self.output_layer(
            x
        )

        loss = None

        # ----------------------------------------------------
        # 6. Next-token language modelling loss
        # ----------------------------------------------------

        if labels is not None:

            # Example:
            #
            # logits:
            # t0 t1 t2
            #
            # targets:
            # t1 t2 t3

            shift_logits = (
                logits[:, :-1, :]
                .contiguous()
            )

            shift_labels = (
                labels[:, 1:]
                .contiguous()
            )

            loss = F.cross_entropy(
                shift_logits.view(
                    -1,
                    shift_logits.size(-1)
                ),
                shift_labels.view(-1)
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits
        )


# Original notebook: a2_partial.ipynb
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # Normalize before attention, as in the original pre-LN block.
        z = self.norm1(x)
        attention_output, _ = self.self_attention(
            z, z, z, key_padding_mask=key_padding_mask, need_weights=False)
        # Add the attention output back to the input (residual connection).
        x = x + self.dropout(attention_output)
        z = self.norm2(x)
        return x + self.dropout(self.feed_forward(z))

class Phase1A2Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, source_length, n_heads, n_layers, d_ff):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.source_position = nn.Embedding(source_length, d_model)
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])

    def forward(self, input_ids, attention_mask):
        seq_len = input_ids.shape[1]
        assert seq_len <= self.source_position.num_embeddings
        # Add a learned position vector to each token embedding.
        pos = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.source_position(pos)
        # Tell attention to ignore padding positions in the batch.
        padding_mask = (attention_mask == 0)
        for block in self.encoder_blocks:
            x = block(x, key_padding_mask=padding_mask)
        return x


# Original notebook: a3_distillation.ipynb
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Compute the RMS in fp32 for numerical stability, then cast back.
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(rms + self.eps).to(x.dtype)
        return x_norm * self.weight

def rotate_half(x):
    # RoPE rotates each consecutive pair: (x0, x1) -> (-x1, x0).
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE needs an even head dimension."
        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def cos_sin(self, positions, dtype):
        # positions: [seq]
        freqs = torch.outer(positions.float(), self.inv_freq)
        cos = freqs.cos().repeat_interleave(2, dim=-1).to(dtype)
        sin = freqs.sin().repeat_interleave(2, dim=-1).to(dtype)
        return cos[None, None, :, :], sin[None, None, :, :]

    def apply_rotary(self, q, k, positions):
        cos, sin = self.cos_sin(positions, q.dtype)
        q = (q * cos) + (rotate_half(q) * sin)
        k = (k * cos) + (rotate_half(k) * sin)
        return q, k

class GroupedQueryLocalAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        window_size,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.groups = num_heads // num_kv_heads
        self.window_size = window_size
        self.chunk_size = chunk_size
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)

        self.rope = RotaryEmbedding(self.head_dim)

    def _repeat_kv(self, x):
        # [B, kv_heads, T, D] -> [B, query_heads, T, D]
        return x.repeat_interleave(self.groups, dim=1)

    def _local_attention(self, q, k, v):
        """
        Exact causal sliding-window attention.

        We process a few query positions at a time.  Each query chunk only
        materializes the K/V region that can actually be attended to, instead
        of creating a full 2048 x 2048 attention matrix.
        """
        B, H, T, D = q.shape
        outputs = []

        k_rep = self._repeat_kv(k)
        v_rep = self._repeat_kv(v)

        for q_start in range(0, T, self.chunk_size):
            q_end = min(q_start + self.chunk_size, T)

            # Earliest key any query in this chunk could legally use.
            k_start = max(0, q_start - self.window_size + 1)
            k_end = q_end

            q_chunk = q[:, :, q_start:q_end, :]
            k_chunk = k_rep[:, :, k_start:k_end, :]
            v_chunk = v_rep[:, :, k_start:k_end, :]

            q_pos = torch.arange(q_start, q_end, device=q.device)
            k_pos = torch.arange(k_start, k_end, device=q.device)

            # True = allowed attention position.
            causal = k_pos[None, :] <= q_pos[:, None]
            within_window = k_pos[None, :] >= (
                q_pos[:, None] - self.window_size + 1
            )
            attn_mask = (causal & within_window)[None, None, :, :]

            out = F.scaled_dot_product_attention(
                q_chunk,
                k_chunk,
                v_chunk,
                attn_mask=attn_mask,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=False,
            )
            outputs.append(out)

        return torch.cat(outputs, dim=2)

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T, _ = x.shape

        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=x.device,
        )
        q, k = self.rope.apply_rotary(q, k, positions)

        if past_kv is None:
            # Training/prefill path: causal local attention over this sequence.
            attn_out = self._local_attention(q, k, v)
            k_all, v_all = k, v
        else:
            # Generation path.  We decode one token at a time after prefill.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv
            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            # Keep only the most recent local window in the KV cache.
            k_all = k_all[:, :, -self.window_size:, :]
            v_all = v_all[:, :, -self.window_size:, :]

            k_rep = self._repeat_kv(k_all)
            v_rep = self._repeat_kv(v_all)

            # Every cached token is in the past (or the current token), so no
            # extra causal mask is necessary for this one-token decode step.
            attn_out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                dropout_p=0.0,
                is_causal=False,
            )

        new_kv = None
        if use_cache:
            new_kv = (
                k_all[:, :, -self.window_size:, :],
                v_all[:, :, -self.window_size:, :],
            )

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.dim)
        return self.o_proj(attn_out), new_kv

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))

class ModernTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        ffn_dim,
        window_size,
        chunk_size,
        dropout,
    ):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = GroupedQueryLocalAttention(
            dim=dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            window_size=window_size,
            chunk_size=chunk_size,
            dropout=dropout,
        )
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_dim, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, position_offset=0):
        attn_out, new_kv = self.attn(
            self.attn_norm(x),
            past_kv=past_kv,
            use_cache=use_cache,
            position_offset=position_offset,
        )
        x = x + self.resid_dropout(attn_out)
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv

class ModernLocalTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=600,
        num_layers=8,
        num_heads=10,
        num_kv_heads=5,
        ffn_dim=1600,
        window_size=512,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        self.num_layers = num_layers

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            ModernTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                num_kv_heads=num_kv_heads,
                ffn_dim=ffn_dim,
                window_size=window_size,
                chunk_size=chunk_size,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Weight tying saves parameters and is standard for causal LMs.
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        x = self.embed_dropout(self.token_embedding(input_ids))
        new_caches = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, new_kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                position_offset=position_offset,
            )
            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)
        return x, new_caches

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        x, _ = self._run_hidden(input_ids)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            # Position t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(loss=loss, logits=logits)

    @torch.inference_mode()
    def prefill(self, input_ids):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )
        return self.lm_head(x), caches

    @torch.inference_mode()
    def decode_step(self, input_ids, past_key_values, position_offset):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )
        return self.lm_head(x[:, -1, :]), caches


# Original notebook: a4_lorma.ipynb
def build_pattern_mask(context_length, pattern_type, pattern_value=None):
    """
    Build a boolean causal attention mask of shape [context_length, context_length].

    PyTorch SDPA interprets True as "this query may attend to this key".

    Example for dilation=2 at query position t:
        allowed keys = t, t-2, t-4, t-6, ...
    """
    query_pos = torch.arange(context_length).unsqueeze(1)  # [T, 1]
    key_pos = torch.arange(context_length).unsqueeze(0)    # [1, T]

    distance = query_pos - key_pos
    causal = distance >= 0

    if pattern_type == "global":
        allowed = causal

    elif pattern_type == "local":
        window = int(pattern_value)
        assert window >= 1
        allowed = causal & (distance < window)

    elif pattern_type == "dilated":
        dilation = int(pattern_value)
        assert dilation >= 1
        allowed = causal & ((distance % dilation) == 0)

    else:
        raise ValueError(f"Unknown attention pattern: {pattern_type}")

    return allowed

class PatternCausalSelfAttention(nn.Module):
    """Causal self-attention with a chosen local/dilated/global pattern."""

    def __init__(
        self,
        dim,
        num_heads,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.context_length = context_length
        self.pattern_type = pattern_type
        self.pattern_value = pattern_value
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

        # The mask is fixed by the architecture, so build it only once.
        # persistent=False means this large mask is NOT written into checkpoints.
        pattern_mask = build_pattern_mask(
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
        )
        self.register_buffer(
            "pattern_mask",
            pattern_mask,
            persistent=False,
        )

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        if past_kv is None:
            # Training / prompt prefill.
            k_all = k
            v_all = v
            key_length = T
            query_start = 0

        else:
            # Incremental generation: append the new token's K/V to the cache.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv

            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            key_length = k_all.size(2)
            query_start = key_length - T

        if key_length > self.context_length:
            raise ValueError(
                f"Attention cache length {key_length} exceeds "
                f"context length {self.context_length}."
            )

        # Slice exactly the query rows and key columns needed in this call.
        # Shape becomes [1, 1, query_length, key_length] and broadcasts over
        # batch and attention heads.
        attn_mask = self.pattern_mask[
            query_start:query_start + T,
            :key_length,
        ][None, None, :, :]

        attn_out = F.scaled_dot_product_attention(
            q,
            k_all,
            v_all,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,  # causality is already encoded in attn_mask
        )

        new_kv = (k_all, v_all) if use_cache else None

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, self.dim)
        )

        return self.o_proj(attn_out), new_kv

class MultiScaleTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        ffn_dim,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()

        # Pre-LayerNorm Transformer block.
        self.norm1 = nn.LayerNorm(dim)
        self.attn = PatternCausalSelfAttention(
            dim=dim,
            num_heads=num_heads,
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
            dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, dim),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False):
        attn_out, new_kv = self.attn(
            self.norm1(x),
            past_kv=past_kv,
            use_cache=use_cache,
        )

        # Residual connection around attention.
        x = x + self.dropout(attn_out)

        # Residual connection around feed-forward network.
        x = x + self.dropout(self.ffn(self.norm2(x)))

        return x, new_kv

class MultiScaleDilatedLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length=2048,
        dim=600,
        num_heads=10,
        ffn_dim=1600,
        attention_patterns=None,
        dropout=0.1,
    ):
        super().__init__()

        if attention_patterns is None:
            raise ValueError("attention_patterns must be provided.")

        self.vocab_size = vocab_size
        self.context_length = context_length
        self.dim = dim
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.attention_patterns = attention_patterns

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(context_length, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            MultiScaleTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                context_length=context_length,
                pattern_type=pattern["type"],
                pattern_value=pattern.get("value"),
                dropout=dropout,
            )
            for pattern in attention_patterns
        ])

        self.final_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Initialize first, then tie token-embedding and output weights.
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T = input_ids.shape

        if position_offset + T > self.context_length:
            raise ValueError(
                f"Sequence position {position_offset + T} exceeds "
                f"context length {self.context_length}."
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=input_ids.device,
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )
        x = self.embed_dropout(x)

        if past_key_values is None:
            layer_past = [None] * len(self.blocks)
        else:
            if len(past_key_values) != len(self.blocks):
                raise ValueError("Wrong number of layer caches.")
            layer_past = past_key_values

        new_caches = []

        for block, past_kv in zip(self.blocks, layer_past):
            x, new_kv = block(
                x,
                past_kv=past_kv,
                use_cache=use_cache,
            )

            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)

        return x, (new_caches if use_cache else None)

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        # Training examples contain full fixed-length token blocks, so no
        # padding mask is needed here. The architectural causal/sparse mask is
        # handled inside each attention layer.
        hidden, _ = self._run_hidden(input_ids)
        logits = self.lm_head(hidden)

        loss = None

        if labels is not None:
            # Next-token language modelling:
            # position t predicts label at position t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits,
        )

    @torch.inference_mode()
    def prefill(self, input_ids):
        """Process the full prompt once and create K/V caches for all layers."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )

        return self.lm_head(hidden), caches

    @torch.inference_mode()
    def decode_step(
        self,
        input_ids,
        past_key_values,
        position_offset,
    ):
        """Process one newly generated token using the existing K/V caches."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )

        logits = self.lm_head(hidden[:, -1, :])
        return logits, caches


# Phase-5 dataset and classification code
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
# These repository paths and weight filenames were read from your prior notebooks.
P1_DEFAULTS = {
    'a1': ('P-261110044/cs781-a1-checkpoints', 'checkpoint-6104/model.safetensors'),
    'a2': ('P-261110044/Phase1-A2', 'cs781_phase1_a2_encoderdecoder/legal-gpt2-final/model_state.pt'),
    'a3': ('P-261110044/cs781-phase1-architecture3', 'model_state.pt'),
    'a4': ('P-261110044/cs781-a4-checkpoints', 'final_model/model_state.pt'),
}


def official_path(split, data_dir=None):
    """Find the public Phase-5 file for a split."""
    rel = f'Phase_5_LSI/{"Test" if split == "test" else "Train"}/{split}.json'
    if data_dir:
        root = Path(data_dir)
        for candidate in [root / rel, root / ('Test' if split == 'test' else 'Train') / f'{split}.json',
                          root / split / f'{split}.json', root / f'{split}.json']:
            if candidate.exists():
                return candidate
        raise FileNotFoundError(f'Could not find {rel} under {root}')
    return Path(hf_hub_download(repo_id=DATA_REPO, repo_type='dataset', filename=rel,
                                token=os.environ.get('HF_TOKEN')))


def read_examples(split, data_dir=None):
    """Read the case text and labels without changing case IDs."""
    raw = json.loads(official_path(split, data_dir).read_text(encoding='utf-8'))
    if isinstance(raw, dict):
        # Accept official list layout, or id->record layout when mirrored locally.
        if isinstance(raw.get('data'), list):
            raw = raw['data']
        elif isinstance(raw.get('examples'), list):
            raw = raw['examples']
        else:
            raw = [dict(v, id=k) if isinstance(v, dict) else {'id': k, 'text': v}
                   for k, v in raw.items()]
    if not isinstance(raw, list):
        raise TypeError(f'{split}: expected list of JSON objects')
    result, seen = [], set()
    for index, ex in enumerate(raw):
        if not isinstance(ex, dict) or 'id' not in ex or 'text' not in ex:
            raise ValueError(f'{split}: record {index} needs id and text')
        ex_id = str(ex['id'])
        if ex_id in seen and split != 'train':
            raise ValueError(f'{split}: duplicate ID {ex_id}')
        text = ex['text']
        if isinstance(text, list): text = '\n'.join(map(str, text))
        if not isinstance(text, str): raise TypeError(f'{ex_id}: text must be string/list')
        labels = ex.get('labels')
        if split != 'test':
            if not isinstance(labels, list): raise ValueError(f'{ex_id}: expected labels list')
            if any(type(x) is not int or x < 0 for x in labels):
                raise ValueError(f'{ex_id}: labels must be nonnegative integers')
            if len(set(labels)) != len(labels):
                raise ValueError(f'{ex_id}: duplicate gold statute ID')
        record = {'id': ex_id, 'text': text,
                  'labels': sorted(set(labels)) if labels is not None else None}
        if ex_id in seen:
            # Some earlier phase splits included repeated training IDs.
            # Exact repeats may be skipped; conflicting records need manual checking.
            old = result[next(i for i, r in enumerate(result) if r['id'] == ex_id)]
            if old != record:
                raise ValueError(f'{split}: conflicting repeated record for {ex_id}')
            continue
        seen.add(ex_id)
        result.append(record)
    if not result: raise ValueError(f'Empty {split} split')
    print(f'Loaded Phase 5 {split}: {len(result)} documents')
    return result


def audit_splits(splits, output_path):
    """Save split fingerprints and report ID/text overlap without using gold test labels."""
    import hashlib
    ids = {name: {r['id'] for r in records} for name,records in splits.items()}
    texts = {name: {hashlib.sha256(r['text'].strip().encode('utf8')).hexdigest()
                    for r in records if r['text'].strip()} for name,records in splits.items()}
    report = {'sizes': {name:len(records) for name,records in splits.items()},
              'sha256': {name: hashlib.sha256(json.dumps(
                   [{'id': r['id'], 'text': r['text'], 'labels': r.get('labels')}
                    for r in records],sort_keys=True).encode('utf8')).hexdigest()
                    for name,records in splits.items()},
              'overlap': {}}
    for a,b in (('train','dev'), ('train','test'), ('dev','test')):
        if a not in splits or b not in splits: continue
        report['overlap'][a+'__'+b] = {'ids':len(ids[a]&ids[b]),
                                     'identical_text':len(texts[a]&texts[b])}
    save_json(output_path, report)
    for pair, item in report['overlap'].items():
        if item['ids'] or item['identical_text']:
            warnings.warn(f'Split overlap {pair}: {item}. Review possible leakage.')
    return report


def get_p1_files(arch, source=None, weight=None, cache_dir=None):
    """Load the Phase-1 checkpoint and tokenizer for an architecture."""
    repo, default_weight = P1_DEFAULTS[arch]
    weight = weight or default_weight
    cache = Path(cache_dir or '.p5_cache') / arch
    cache.mkdir(parents=True, exist_ok=True)
    if source and Path(source).is_dir():
        folder = Path(source)
        files = [str(p.relative_to(folder)) for p in folder.rglob('*') if p.is_file()]
        def grab(rel): return folder / rel
        provenance = str(folder.resolve())
    else:
        repo = source or repo
        client = HfApi(token=os.environ.get('HF_TOKEN'))
        files = client.list_repo_files(repo_id=repo, repo_type='model')
        def grab(rel):
            return Path(hf_hub_download(repo_id=repo, repo_type='model', filename=rel,
                                        token=os.environ.get('HF_TOKEN')))
        provenance = repo
    if weight not in files:
        raise FileNotFoundError(f'Expected {weight!r} in Phase 1 source {provenance}; '
                                'check --source / --weight rather than using different model weights.')
    wf = grab(weight)
    # Tokenizer is chosen from same checkpoint directory first, then known roots.
    folder = str(Path(weight).parent).replace('\\', '/')
    folder = '' if folder == '.' else folder + '/'
    ordered = list(dict.fromkeys([folder, '', 'final_model/',
                                  'cs781_phase1_a2_encoderdecoder/legal-gpt2-final/']))
    tokbase = next((p for p in ordered if p + 'tokenizer.json' in files), None)
    if tokbase is None: raise FileNotFoundError(f'No Phase 1 tokenizer.json in {provenance}')
    # Keep the tokenizer from the same Phase 1 checkpoint.
    tdir = cache / 'tokenizer'
    tdir.mkdir(parents=True, exist_ok=True)
    for name in ('tokenizer.json','tokenizer_config.json','special_tokens_map.json',
                 'added_tokens.json'):
        if tokbase + name in files:
            (tdir / name).write_bytes(grab(tokbase + name).read_bytes())
    if PreTrainedTokenizerFast is None:
        raise ImportError('Install transformers and tokenizers: pip install transformers tokenizers')
    # Load the same tokenizer.json used during Phase 1. Never re-train or extend it.
    tokenizer_file = tdir / 'tokenizer.json'
    if not tokenizer_file.is_file():
        raise FileNotFoundError(f'Missing original tokenizer file: {tokenizer_file}')
    from tokenizers import Tokenizer as RawTokenizer
    raw_vocabulary = RawTokenizer.from_file(str(tokenizer_file)).get_vocab(with_added_tokens=True)
    special = {'pad_token': '<pad>', 'unk_token': '<unk>',
               'bos_token': '<s>', 'eos_token': '</s>'}
    absent = [value for value in special.values() if value not in raw_vocabulary]
    if absent:
        raise ValueError('Tokenizer is missing Phase 1 special tokens '+str(absent)+
                         '. Check --source/--weight; do not add new token IDs.')
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=str(tokenizer_file), **special)
    if not tokenizer.is_fast:
        raise RuntimeError('Expected a fast Phase 1 tokenizer')
    for name, value in special.items():
        if getattr(tokenizer, name + '_id') != raw_vocabulary[value]:
            raise ValueError(f'Special token {value!r} does not match Phase 1 token ID')
    configloc = next((x for x in [folder+'architecture_config.json',
                                  'architecture_config.json'] if x in files), None)
    cfg = json.loads(grab(configloc).read_text()) if configloc else {}
    if str(wf).endswith('.safetensors'):
        from safetensors.torch import load_file
        state = load_file(str(wf), device='cpu')
    else:
        state = torch.load(str(wf), map_location='cpu', weights_only=True)
    for key in ('model_state', 'state_dict', 'model_state_dict'):
        if isinstance(state, dict) and key in state and isinstance(state[key], dict):
            state = state[key]
    for prefix in ('module.', '_orig_mod.'):
        if all(k.startswith(prefix) for k in state):
            state = {k[len(prefix):]: v for k,v in state.items()}
    if 'token_embedding.weight' not in state: raise ValueError('Unexpected checkpoint format: missing token_embedding.weight')
    vocab, dim = state['token_embedding.weight'].shape
    if vocab != len(tokenizer):
        raise ValueError(f'Phase 1 tokenizer mismatch: {len(tokenizer)} vs {vocab}')
    return state, tokenizer, cfg, {'arch': arch, 'source': provenance, 'weight': weight}


def count_blocks(state, key):
    indices = {int(m.group(1)) for name in state
               if (m := re.match(r'^'+re.escape(key)+r'\.(\d+)\.', name))}
    if not indices or indices != set(range(max(indices)+1)):
        raise ValueError(f'Incomplete pretrained layers: {key}')
    return len(indices)


def build_pretrained_backbone(arch, state, cfg, heads_override=None, num_layers=None):
    vocab, dim = state['token_embedding.weight'].shape
    if arch == 'a1':
        layers = count_blocks(state,'blocks')
        nh = int(heads_override or cfg.get('n_heads',cfg.get('num_heads',10)))
        if dim % nh: raise ValueError('A1 head count not compatible with checkpoint dimension')
        options=dict(vocab_size=vocab, context_length=state['position_embedding.weight'].shape[0],
                     d_model=dim, n_heads=nh, n_layers=layers if num_layers is None else num_layers,
                     d_ff=state['blocks.0.feed_forward.0.weight'].shape[0],dropout=0.1)
        base=DecoderTransformerLM(**options)
    elif arch == 'a2':
        layers = count_blocks(state,'encoder_blocks')
        nh = int(heads_override or cfg.get('n_heads',cfg.get('num_heads',15 if dim==600 else 8)))
        if dim % nh: raise ValueError('A2 head count not compatible with checkpoint')
        options=dict(vocab_size=vocab, d_model=dim,
                     source_length=state['source_position.weight'].shape[0],
                     n_heads=nh, n_layers=layers if num_layers is None else num_layers,
                     d_ff=state['encoder_blocks.0.feed_forward.0.weight'].shape[0])
        base=Phase1A2Encoder(**options)
    elif arch=='a3':
        required=('num_heads','num_kv_heads','window_size','chunk_size')
        if any(k not in cfg for k in required):
            raise ValueError('A3 requires architecture_config.json with '+str(required))
        layers=count_blocks(state,'blocks')
        options=dict(vocab_size=vocab,dim=dim,
                     num_layers=layers if num_layers is None else num_layers,
                     num_heads=int(cfg['num_heads']),num_kv_heads=int(cfg['num_kv_heads']),
                     ffn_dim=state['blocks.0.ffn.gate_proj.weight'].shape[0],
                     window_size=int(cfg['window_size']),chunk_size=int(cfg['chunk_size']),
                     dropout=float(cfg.get('dropout',0.1)))
        base=ModernLocalTransformerLM(**options)
    elif arch=='a4':
        if 'attention_patterns' not in cfg or 'num_heads' not in cfg:
            raise ValueError('A4 requires saved attention_patterns and num_heads')
        layers=count_blocks(state,'blocks')
        patterns=cfg['attention_patterns']
        if len(patterns)!=layers: raise ValueError('A4 attention mask pattern/layer mismatch')
        options=dict(vocab_size=vocab,context_length=state['position_embedding.weight'].shape[0],
                     dim=dim,num_heads=int(cfg['num_heads']),
                     ffn_dim=state['blocks.0.ffn.0.weight'].shape[0],
                     attention_patterns=patterns[:num_layers] if num_layers else patterns,
                     dropout=float(cfg.get('dropout',0.1)))
        base=MultiScaleDilatedLM(**options)
    else: raise ValueError('Unknown architecture '+str(arch))
    # Keep alias handling from your Phase 2 code; no unexpected keys allowed.
    expected = base.state_dict()
    if arch=='a2':
        selected={k:v for k,v in state.items() if k in expected}
    else:
        selected=dict(state)
        if arch in ('a3','a4'):
            for key,alt in (('lm_head.weight','token_embedding.weight'),
                            ('token_embedding.weight','lm_head.weight')):
                if key in expected and key not in selected and alt in selected:
                    selected[key]=selected[alt]
        if arch=='a1' and 'lm_head.weight' in selected and 'lm_head.weight' not in expected:
            selected.setdefault('output_layer.weight',selected['lm_head.weight'])
            del selected['lm_head.weight']
    if num_layers is not None:
        # For student distillation, extra pretrained layers are intentionally excluded.
        selected = {k:v for k,v in selected.items() if k in expected}
    missing=set(expected)-set(selected)
    extra=set(selected)-set(expected)
    wrong=[k for k in expected.keys() & selected.keys() if expected[k].shape != selected[k].shape]
    if missing or extra or wrong:
        raise ValueError(f'{arch} strict pretrained-weight mismatch: missing={list(missing)[:5]}, '
                         f'extra={list(extra)[:5]}, wrong={wrong[:5]}')
    base.load_state_dict(selected,strict=True)
    if arch=='a1': base.output_layer=nn.Identity()  # Return hidden states without vocabulary logits.
    if arch=='a2' and num_layers is None:
        # All original encoder parameters, no randomly initialized encoder block.
        assert len(selected)==len(expected)
    return base, dim, options


class LoRALinear(nn.Module):
    """LoRA update added to the frozen linear layer."""
    def __init__(self, original, rank=8, alpha=16):
        super().__init__()
        self.original=original
        self.original.requires_grad_(False)
        self.A=nn.Linear(original.in_features, rank,bias=False)
        self.B=nn.Linear(rank,original.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=0.02)
        nn.init.zeros_(self.B.weight)
        self.scale=alpha/rank
    def forward(self,x): return self.original(x)+self.B(self.A(x))*self.scale


class LoRMALinear(nn.Module):
    """Apply (I + BA) to the output of the frozen layer."""
    def __init__(self, original,rank=8,alpha=16):
        super().__init__()
        self.original=original
        self.original.requires_grad_(False)
        self.A=nn.Linear(original.out_features,rank,bias=False)
        self.B=nn.Linear(rank,original.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=0.02)
        nn.init.zeros_(self.B.weight)
        self.scale=alpha/rank
    def forward(self,x):
        y=self.original(x)
        return y+self.B(self.A(y))*self.scale


class ResidualAdapter(nn.Module):
    """Small adapter after a transformer block."""
    def __init__(self, dim, bottleneck=64):
        super().__init__()
        self.down=nn.Linear(dim,bottleneck)
        self.up=nn.Linear(bottleneck,dim)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self,hidden):return hidden+self.up(F.gelu(self.down(hidden)))


class StatuteModel(nn.Module):
    """Predict statute scores from several windows of the judgment."""
    def __init__(self, backbone, arch, dim, num_labels, method='full',rank=8,alpha=16):
        super().__init__()
        self.backbone=backbone
        self.arch=arch
        self.method=method
        self.dropout=nn.Dropout(.15)
        self.classifier=nn.Linear(dim*2,num_labels) # mean+max across windows
        if method not in ('full','kd'): self.backbone.requires_grad_(False)
        if method=='partial':
            blocks=self.backbone.encoder_blocks if arch=='a2' else self.backbone.blocks
            for block in blocks[-2:]: block.requires_grad_(True)
        elif method in ('lora','lorma'):
            cls=LoRALinear if method=='lora' else LoRMALinear
            count=0
            for block in self.backbone.blocks:
                attention = getattr(block,'attn',None)
                if attention is None: continue
                for proj in ('q_proj','v_proj'):
                    old=getattr(attention,proj,None)
                    if isinstance(old,nn.Linear):
                        setattr(attention,proj,cls(old,rank,alpha))
                        count+=1
            if not count: raise ValueError(method+' requires q_proj/v_proj; use A3/A4')
        elif method=='adapters':
            if arch!='a1': raise ValueError('Adapters configured for A1 transformer blocks')
            self.adapters=nn.ModuleList([ResidualAdapter(dim) for _ in self.backbone.blocks])
            # Hook each block output using a closure that captures the corresponding adapter.
            for block,adapter in zip(self.backbone.blocks,self.adapters):
                block.register_forward_hook(lambda mod,inp,out,ad=adapter:ad(out))
        elif method not in ('full','kd','partial'):
            raise ValueError(f'Unknown adaptation method {method}')

    def token_hidden(self, ids,mask):
        if self.arch=='a1': return self.backbone(input_ids=ids).logits
        if self.arch=='a2': return self.backbone(ids,mask)
        return self.backbone._run_hidden(ids)[0]

    def forward(self,ids,mask):
        # ids shape [batch, chunks, tokens]. Flatten for pretrained window encoding.
        batch,chunks,tokens=ids.shape
        states=self.token_hidden(ids.reshape(-1,tokens),mask.reshape(-1,tokens))
        token_mask=mask.reshape(-1,tokens).unsqueeze(-1).to(states.dtype)
        pooled=(states*token_mask).sum(1)/token_mask.sum(1).clamp_min(1)
        pooled=pooled.reshape(batch,chunks,-1)
        # Multiple evidence windows; max finds decisive statute mentions.
        valid=(mask.sum(-1)>0)
        maximum=pooled.masked_fill(~valid.unsqueeze(-1),torch.finfo(pooled.dtype).min).max(1).values
        average=(pooled*valid.unsqueeze(-1)).sum(1)/valid.sum(1,keepdim=True).clamp_min(1)
        return self.classifier(self.dropout(torch.cat([average,maximum],dim=-1)))


STATUTE_RE = re.compile(r'(?i)\b(?:section|sec\.|article|art\.|subsection|sub-section|act\s+of|'
                         r'provision|statute|code|rule\s+\d|chapter\s+\w|ipc|crpc)\b')


def evidence_texts(text,max_chunks=3,chunk_chars=1800):
    """Keep the beginning, an evidence window and the ending."""
    if not text: return ['']
    if len(text) <= chunk_chars: return [text]
    result=[text[:chunk_chars]]
    if max_chunks >= 3:
        # Middle evidence: dense citation window, selected using text only (not labels).
        middle=[]
        for match in STATUTE_RE.finditer(text):
            if chunk_chars < match.start() < len(text)-chunk_chars:
                middle.append(match.start())
            if len(middle)>=40:break
        center=middle[len(middle)//2] if middle else len(text)//2
        start=max(0,min(len(text)-chunk_chars,center-chunk_chars//2))
        result.append(text[start:start+chunk_chars])
    if max_chunks >= 2:result.append(text[-chunk_chars:])
    return result[:max_chunks]


def tokenize_examples(records,tokenizer,max_length=256,max_chunks=3):
    """Tokenize each selected window once."""
    if max_length < 2 or max_chunks < 1:
        raise ValueError('max_length must be >=2 and max_chunks must be >=1')
    output=[]
    for record in records:
        windows=evidence_texts(record['text'],max_chunks=max_chunks)
        # Repeat last real window instead of feeding an all-pad chunk to MHA.
        while len(windows) < max_chunks: windows.append(windows[-1])
        encoded=tokenizer(windows,padding='max_length',truncation=True,
                          max_length=max_length,return_attention_mask=True)
        output.append({'id':record['id'],
                       'ids':torch.tensor(encoded['input_ids'],dtype=torch.long),
                       'mask':torch.tensor(encoded['attention_mask'],dtype=torch.long)})
    return output


def labels_matrix(examples,label_ids):
    to_index={lab:i for i,lab in enumerate(label_ids)}
    y=np.zeros((len(examples),len(label_ids)),dtype=np.float32)
    for row,ex in enumerate(examples):
        for label in ex['labels'] or []:
            if label in to_index:y[row,to_index[label]]=1
    return y


def label_vocabulary(train,dev):
    # Collect the available statute IDs. Dev labels are not used in the loss.
    return sorted(set(x for ex in train+dev for x in ex['labels']))


def apply_thresholds(probs,thresholds,ensure_one=False):
    thresholds=np.asarray(thresholds)
    pred=probs >= (thresholds if thresholds.ndim==0 else thresholds[None,:])
    if ensure_one:
        for i in range(len(pred)):
            if not pred[i].any():pred[i,int(np.argmax(probs[i]))]=True
    return pred.astype(np.int8)


def multilabel_metrics(y,p):
    """sklearn-compatible macro F1 on the explicitly provided label list."""
    from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
    return {'macro_f1':float(f1_score(y,p,average='macro',zero_division=0)),
            'micro_f1':float(f1_score(y,p,average='micro',zero_division=0)),
            'subset_accuracy':float(accuracy_score(y,p)),
            'macro_precision':float(precision_score(y,p,average='macro',zero_division=0)),
            'macro_recall':float(recall_score(y,p,average='macro',zero_division=0))}


def calibrate_thresholds(y,p,train_support):
    """Find a shared threshold and smooth per-label thresholds."""
    from sklearn.metrics import f1_score
    # Smaller thresholds may help labels with low predicted probabilities.
    choices=np.arange(.05,.61,.025)
    candidates=[(float(f1_score(y,apply_thresholds(p,t),average='macro',zero_division=0)),float(t))
                for t in choices]
    global_thr=max(candidates,key=lambda v:v[0])[1]
    result=np.full(y.shape[1],global_thr,dtype=np.float32)
    for j in range(y.shape[1]):
        positive=int(y[:,j].sum())
        if positive < 3: continue # tiny validation support -> unreliable individual F1
        scored=[(float(f1_score(y[:,j],p[:,j]>=t,zero_division=0)),float(t)) for t in choices]
        best=max(scored,key=lambda v:v[0])[1]
        # Keep rare-label thresholds closer to the shared value.
        shrink=positive/(positive+8.0)
        result[j]=shrink*best+(1-shrink)*global_thr
    return result,{'global_threshold':global_thr,'global_search_macro_f1':max(x[0] for x in candidates)}


def save_json(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(data,indent=2,ensure_ascii=False),encoding='utf-8')


def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

class LexicalKNN:
    """TF-IDF neighbour votes fitted on the Phase-5 training split."""
    def __init__(self,k=5):
        self.k=k

    @staticmethod
    def abbreviated(text):
        # Include both ends and the same middle evidence used in transformer windows.
        return ' '.join(evidence_texts(text,max_chunks=3,chunk_chars=1600))

    def fit(self,texts,targets):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.neighbors import NearestNeighbors
        self.vectorizer=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,
              max_features=40000,min_df=1,strip_accents='unicode',dtype=np.float32)
        self.vectors=self.vectorizer.fit_transform([self.abbreviated(x) for x in texts])
        self.targets=np.asarray(targets,dtype=np.float32)
        self.nn=NearestNeighbors(n_neighbors=min(self.k,len(texts)),metric='cosine',algorithm='brute',n_jobs=-1)
        self.nn.fit(self.vectors)
        return self

    def predict(self,texts,batch_size=32):
        results=[]
        for start in range(0,len(texts),batch_size):
            batch=[self.abbreviated(t) for t in texts[start:start+batch_size]]
            features=self.vectorizer.transform(batch)
            dist,neighbors=self.nn.kneighbors(features)
            sim=np.maximum(1-dist,0)**2  # Emphasize genuinely similar cases.
            sums=sim.sum(axis=1,keepdims=True)
            vote=(self.targets[neighbors]*sim[:,:,None]).sum(axis=1)/np.maximum(sums,1e-6)
            results.append(vote)
        return np.concatenate(results).astype(np.float32)


Writing phase5_core.py


In [4]:
from pathlib import Path

core_file = WORKDIR / "phase5_core.py"
code = core_file.read_text(encoding="utf-8")

old = """            if len(set(labels)) != len(labels):
                raise ValueError(f'{ex_id}: duplicate gold statute ID')"""

new = """            # Keep each statute ID once.
            labels = sorted(set(labels))"""

if old in code:
    code = code.replace(old, new)
    core_file.write_text(code, encoding="utf-8")
    print("Duplicate statute label handling fixed.")
elif new in code:
    print("Fix already applied.")
else:
    raise RuntimeError("Expected code not found. Check phase5_core.py.")

compile(core_file.read_text(encoding="utf-8"),
        str(core_file), "exec")

print("Syntax check passed. Ready to train.")

Duplicate statute label handling fixed.
Syntax check passed. Ready to train.


In [5]:
%%writefile phase5_train.py
#!/usr/bin/env python3
"""Train the Phase-5 multi-label classifier from a Phase-1 checkpoint.

Examples:
 python phase5_train.py --arch a1 --method full --out runs/a1_full
 python phase5_train.py --arch a2 --method partial --out runs/a2_partial
 python phase5_train.py --arch a3 --method lora --out runs/a3_lora
 python phase5_train.py --arch a4 --method lorma --out runs/a4_lorma
 python phase5_train.py --arch a1 --method adapters --out runs/a1_adapters
"""
import argparse, json, time
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from phase5_core import (get_p1_files,build_pretrained_backbone,StatuteModel,
    read_examples,tokenize_examples,labels_matrix,label_vocabulary,audit_splits,
    calibrate_thresholds,apply_thresholds,multilabel_metrics,save_json,set_seed,LexicalKNN)
from phase5_official_bridge import (valid_dev_columns, score_dev_probability_predictions,
                                    write_dev_gold, run_official)


def parse_args():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--arch',required=True,choices=['a1','a2','a3','a4'])
    p.add_argument('--method',required=True,choices=['full','partial','lora','lorma','adapters'])
    p.add_argument('--source',default=None,help='Phase 1 HF repo ID or existing local folder')
    p.add_argument('--weight',default=None,help='Exact Phase 1 weight path inside source')
    p.add_argument('--data-dir',default=None,help='Local Phase_5_LSI dir; omitted downloads official dataset')
    p.add_argument('--out',required=True,help='Unique experiment output directory')
    p.add_argument('--epochs',type=int,default=5)
    p.add_argument('--batch-size',type=int,default=1)
    p.add_argument('--accum',type=int,default=4)
    p.add_argument('--length',type=int,default=256)
    p.add_argument('--chunks',type=int,default=3)
    p.add_argument('--lr',type=float,default=None)
    p.add_argument('--head-lr',type=float,default=2e-4)
    p.add_argument('--max-train',type=int,default=None,help='Smoke-test cap; do not use for final runs')
    p.add_argument('--max-dev',type=int,default=None,help='Smoke-test cap; do not use for final runs')
    p.add_argument('--seed',type=int,default=44)
    p.add_argument('--lexical-knn',action='store_true',
                   help='Compare optional train-only TF-IDF nearest-neighbour blend on dev')
    p.add_argument('--heads',type=int,default=None,help='Only if A1/A2 checkpoint requires manual head count')
    p.add_argument('--rank',type=int,default=8)
    p.add_argument('--alpha',type=float,default=16)
    return p.parse_args()


def eval_probs(model,loader,device):
    model.eval(); scores=[]
    with torch.inference_mode():
        for batch in loader:
            logits=model(batch['ids'].to(device),batch['mask'].to(device))
            scores.append(logits.float().sigmoid().cpu().numpy())
    return np.concatenate(scores,axis=0)


def dataset(records,tokenizer,args):
    return tokenize_examples(records,tokenizer,args.length,args.chunks)


def choose_thresholds(y,probs,support,examples=None,label_ids=None):
    # Test the usual cutoff and two cutoffs based on dev predictions.
    per_label,meta=calibrate_thresholds(y,probs,support)
    global_thr=meta['global_threshold']
    if examples is not None:
        if label_ids is None:
            raise ValueError('Pass label_ids along with dev examples')
        # The course scorer forces at least one predicted label per example here.
        global_thr=max((float(t) for t in np.arange(.05,.61,.025)),
            key=lambda t:score_dev_probability_predictions(examples,label_ids,probs,t)[0]['macro_f1'])
        # Use the same globally selected cutoff for the smoothed per-label thresholds.
        per_label=np.clip(per_label + (global_thr - meta['global_threshold']), .02, .95)
        meta['global_threshold']=float(global_thr)
    alternatives={
        'default_0_5':np.full(y.shape[1],.5,dtype=np.float32),
        'calibrated_global':np.full(y.shape[1],global_thr,dtype=np.float32),
        'calibrated_per_label':per_label,
    }
    if examples is not None:
        if label_ids is None:
            raise ValueError('Pass label_ids along with dev examples')
        # This uses the same label dimension as the course evaluator.
        scored={name:score_dev_probability_predictions(examples,label_ids,probs,th)[0]
                for name,th in alternatives.items()}
    else:
        scored={name:multilabel_metrics(y,apply_thresholds(probs,th,ensure_one=True))
                for name,th in alternatives.items()}
    best=max(scored,key=lambda name:scored[name]['macro_f1'])
    return alternatives[best], {'selected':best,'scores':scored,**meta}


def main():
    a=parse_args(); set_seed(a.seed)
    if a.method=='lora' and a.arch not in ('a3','a4'): raise ValueError('LoRA uses A3/A4 q_proj/v_proj')
    if a.method=='lorma' and a.arch not in ('a3','a4'): raise ValueError('LoRMA uses A3/A4 q_proj/v_proj')
    if a.method=='adapters' and a.arch!='a1': raise ValueError('This adapter implementation uses A1')
    out=Path(a.out);out.mkdir(parents=True,exist_ok=True)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device:',device,'Training:',a.arch,a.method)
    train=read_examples('train',a.data_dir)
    dev=read_examples('dev',a.data_dir)
    audit_splits({'train': train, 'dev': dev}, out / 'data_audit.json')
    if a.epochs < 1 or a.batch_size < 1 or a.accum < 1:
        raise ValueError('--epochs, --batch-size and --accum must be positive')
    # Only the train split is used for gradient updates.
    if a.max_train:train=train[:a.max_train]
    if a.max_dev:dev=dev[:a.max_dev]
    label_ids=label_vocabulary(train,dev)
    y_train=labels_matrix(train,label_ids)
    y_dev=labels_matrix(dev,label_ids)
    support=y_train.sum(0)
    # Dev scoring rejects IDs larger than its largest gold label.
    # Keep those extra IDs in the model for test prediction.
    dev_cols=valid_dev_columns(dev,label_ids)
    dev_labels=[lab for lab,ok in zip(label_ids,dev_cols) if ok]
    dev_targets=y_dev[:,dev_cols]
    dev_support=support[dev_cols]
    dev_gold_path=out/'dev_gold.json'
    print('Official development label dimension:', write_dev_gold(dev,dev_gold_path))
    if not label_ids:raise RuntimeError('No statute labels found in public train/dev')
    print('Label vocabulary:',len(label_ids),'train-positive count:',int(y_train.sum()))
    state,tok,orig_cfg,provenance=get_p1_files(a.arch,a.source,a.weight,out/'hf_cache')
    base,dim,options=build_pretrained_backbone(a.arch,state,orig_cfg,a.heads)
    # Torch state on CPU can be freed before moving the backbone to GPU.
    del state
    context=int(options.get('context_length',options.get('source_length',2048)))
    if a.length>context:raise ValueError(f'--length {a.length} > pretrained context {context}')
    model=StatuteModel(base,a.arch,dim,len(label_ids),a.method,a.rank,a.alpha).to(device)
    all_params=sum(x.numel() for x in model.parameters())
    active=sum(x.numel() for x in model.parameters() if x.requires_grad)
    print(f'Parameters total {all_params:,}, trainable {active:,}')
    tdata=dataset(train,tok,a); ddata=dataset(dev,tok,a)
    train_by_id={record['id']:idx for idx,record in enumerate(train)}
    trloader=DataLoader(tdata,batch_size=a.batch_size,shuffle=True)
    dvloader=DataLoader(ddata,batch_size=a.batch_size,shuffle=False)
    base_lr=a.lr or (2e-5 if a.method=='full' else 8e-5)
    backbone_trainable=[p for n,p in model.named_parameters()
                        if p.requires_grad and not n.startswith('classifier.')]
    optim=torch.optim.AdamW([{'params':backbone_trainable,'lr':base_lr},
                             {'params':model.classifier.parameters(),'lr':a.head_lr}],weight_decay=0.01)
    # Every statute is predicted independently; cap large rare-label weights.
    pos_weight=np.sqrt((len(train)-support+1)/(support+1))
    pos_weight=np.clip(pos_weight,1,12)
    # A label with no training positives still gets a finite weight.
    loss_fn=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight,device=device,dtype=torch.float32))
    lex_dev=None
    if a.lexical_knn:
        # No outside data: sparse TF-IDF vectors and label targets from official train only.
        import joblib
        lexical=LexicalKNN(k=5).fit([x['text'] for x in train],y_train)
        lex_dev=lexical.predict([x['text'] for x in dev])
        joblib.dump(lexical,out/'lexical.joblib')
        print('Optional lexical kNN dev probabilities cached')
    config={'arch':a.arch,'method':a.method,'options':options,'phase1':provenance,
            'labels':label_ids,'length':a.length,'chunks':a.chunks,'rank':a.rank,
            'alpha':a.alpha,'head_lr':a.head_lr,'lr':base_lr,
            'seed':a.seed,'parameters_total':all_params,'parameters_trainable':active}
    save_json(out/'config.json',config)
    # Use mixed precision on a GPU when supported.
    use_cuda=device.type=='cuda'
    use_bf16=use_cuda and torch.cuda.is_bf16_supported()
    scaler=torch.amp.GradScaler('cuda',enabled=use_cuda and not use_bf16)
    best=-1.;history=[]
    for epoch in range(1,a.epochs+1):
        start=time.time();model.train();running=0.;optim.zero_grad(set_to_none=True)
        for step,batch in enumerate(trloader,1):
            # Match labels to IDs after DataLoader shuffling.
            target=np.stack([y_train[train_by_id[k]] for k in batch['id']])
            target=torch.tensor(target,device=device,dtype=torch.float32)
            with torch.autocast('cuda',dtype=torch.bfloat16 if use_bf16 else torch.float16,
                                enabled=use_cuda):
                logits=model(batch['ids'].to(device),batch['mask'].to(device))
                loss=loss_fn(logits.float(),target)
            scaler.scale(loss/a.accum).backward()
            if step%a.accum==0 or step==len(trloader):
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                scaler.step(optim);scaler.update()
                optim.zero_grad(set_to_none=True)
            running+=float(loss.item())
        probs=eval_probs(model,dvloader,device)
        # Compare cutoffs and, if enabled, a TF-IDF blend on dev.
        candidate_weights=(0.,.25,.5,.75) if lex_dev is not None else (0.,)
        decisions=[]
        for mix in candidate_weights:
            combined=(1-mix)*probs+mix*lex_dev if lex_dev is not None else probs
            # Limit dev predictions to IDs accepted by the official scorer.
            t,info=choose_thresholds(dev_targets,combined[:,dev_cols],dev_support,dev,dev_labels)
            m,candidate_preds=score_dev_probability_predictions(
                dev,dev_labels,combined[:,dev_cols],t)
            decisions.append((m['macro_f1'],mix,t,info,m,candidate_preds))
        _,selected_mix,threshold,cal,metrics,dev_pred=max(decisions,key=lambda d:d[0])
        config['lexical_weight']=selected_mix
        row={'epoch':epoch,'train_loss':round(running/max(1,len(trloader)),6),
             'seconds':round(time.time()-start,1),**metrics,'calibration':cal}
        history.append(row);save_json(out/'history.json',history)
        print('epoch',epoch,'loss',row['train_loss'],'dev macro-F1',round(metrics['macro_f1'],5),
              'dev micro-F1',round(metrics['micro_f1'],5))
        if metrics['macro_f1']>best:
            best=metrics['macro_f1']
            save_json(out/'config.json',config)
            # Keep one cutoff for IDs outside the dev scorer's range.
            test_thresholds=np.full(len(label_ids),cal['global_threshold'],dtype=np.float32)
            test_thresholds[dev_cols]=threshold
            save_json(out/'dev_prediction.json',dev_pred)
            # Run the supplied Phase-5 evaluator on the saved dev predictions.
            run_official(dev_gold_path,out/'dev_prediction.json')
            torch.save({'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                        'config':config,'thresholds':test_thresholds.tolist(),
                        'best_dev_metrics':metrics},out/'best.pt')
            save_json(out/'best_metrics.json',{'epoch':epoch,**metrics,'calibration':cal})
    print('BEST held-out Macro-F1:',best,'checkpoint:',out/'best.pt')


if __name__=='__main__':
    main()


Writing phase5_train.py


In [6]:
%%writefile phase5_predict.py
#!/usr/bin/env python3
"""Write prediction.json and submission.zip for a trained Phase-5 run.

 python phase5_predict.py --run runs/a4_lorma --out submissions/a4_lorma
"""
import argparse,json,zipfile
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from phase5_core import (get_p1_files,build_pretrained_backbone,StatuteModel,
    read_examples,tokenize_examples,apply_thresholds,save_json)


def load_trained(run,source=None,weight=None,device=None):
    run=Path(run)
    if not (run/'best.pt').is_file():
        raise FileNotFoundError(f'Missing trained checkpoint: {run/"best.pt"}')
    ckpt=torch.load(run/'best.pt',map_location='cpu',weights_only=True)
    cfg=ckpt['config'];arch=cfg['arch'];method=cfg['method']
    # Recreate the backbone before loading the task checkpoint.
    state,tok,p1_cfg,provenance=get_p1_files(arch,source or cfg['phase1']['source'],
                                           weight or cfg['phase1']['weight'],run/'predict_cache')
    layers=cfg.get('student_layers')
    base,dim,_=build_pretrained_backbone(arch,state,p1_cfg,
                                        cfg['options'].get('n_heads') if arch in ('a1','a2') else None,
                                        num_layers=layers)
    del state
    model=StatuteModel(base,arch,dim,len(cfg['labels']),method,
                       cfg.get('rank',8),cfg.get('alpha',16))
    model.load_state_dict(ckpt['model'],strict=True)
    cutoff=np.asarray(ckpt['thresholds'],dtype=np.float32)
    if cutoff.shape != (len(cfg['labels']),) or not np.all(np.isfinite(cutoff)):
        raise ValueError('Checkpoint thresholds are missing or do not match statute IDs')
    model=model.to(device or ('cuda' if torch.cuda.is_available() else 'cpu')).eval()
    return model,tok,cfg,cutoff


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--run',required=True,help='Training experiment directory containing best.pt')
    p.add_argument('--out',required=True,help='Directory for prediction.json and submission.zip')
    p.add_argument('--data-dir',default=None)
    p.add_argument('--source',default=None,help='Optional override if pretrained repo not accessible')
    p.add_argument('--weight',default=None)
    p.add_argument('--batch-size',type=int,default=1)
    a=p.parse_args()
    out=Path(a.out);out.mkdir(parents=True,exist_ok=True)
    model,tok,cfg,threshold=load_trained(a.run,a.source,a.weight)
    tests=read_examples('test',a.data_dir)
    if a.batch_size < 1: raise ValueError('--batch-size must be >=1')
    examples=tokenize_examples(tests,tok,cfg['length'],cfg['chunks'])
    loader=DataLoader(examples,batch_size=a.batch_size,shuffle=False)
    scores=[];device=next(model.parameters()).device
    with torch.inference_mode():
        for b in loader:
            # Calculate probabilities from the classifier outputs.
            probs=model(b['ids'].to(device),b['mask'].to(device)).float().sigmoid()
            scores.append(probs.cpu().numpy())
    probs=np.concatenate(scores,axis=0)
    if cfg.get('lexical_weight',0)>0:
        import joblib
        lexical=joblib.load(Path(a.run)/'lexical.joblib')
        lex_prob=lexical.predict([x['text'] for x in tests])
        mix=float(cfg['lexical_weight'])
        probs=(1-mix)*probs+mix*lex_prob
    binary=apply_thresholds(probs,threshold,ensure_one=True)
    preds=[]
    for example,flags in zip(tests,binary):
        statute_ids=[int(cfg['labels'][j]) for j in np.flatnonzero(flags)]
        # Deduplicate and sort IDs before writing JSON.
        preds.append({'id':example['id'],'pred':sorted(set(statute_ids))})
    assert len(preds)==len(tests) and len({x['id'] for x in preds})==len(tests)
    assert all(r['pred'] and set(r['pred']).issubset(set(cfg['labels'])) for r in preds)
    save_json(out/'prediction.json',preds)
    with zipfile.ZipFile(out/'submission.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(out/'prediction.json',arcname='prediction.json')
    print('Wrote',out/'prediction.json','and',out/'submission.zip')
    print('Documents:',len(preds),'mean predicted statutes:',round(binary.sum(1).mean(),3))


if __name__=='__main__':main()


Writing phase5_predict.py


In [7]:
%%writefile phase5_validate.py
#!/usr/bin/env python3
"""Check Phase-5 submission files and score public dev predictions.

 python phase5_validate.py --self-test
 python phase5_validate.py --pred submissions/a4_lorma/submission.zip --split test
 python phase5_validate.py --pred dev_prediction.json --split dev

Dev metrics are delegated to the user-provided official evaluator.
ZIP structure/ID checks are additional integrity checks.
"""
import argparse,io,json,zipfile,tempfile
from pathlib import Path
import numpy as np
from phase5_core import (read_examples,label_vocabulary,labels_matrix,multilabel_metrics)
from phase5_official_bridge import write_dev_gold,run_official


def load_predictions(path):
    p=Path(path)
    if p.suffix.lower()=='.zip':
        with zipfile.ZipFile(p,'r') as archive:
            names=archive.namelist()
            if names != ['prediction.json']:
                raise ValueError('ZIP must contain prediction.json at archive ROOT only: '+str(names))
            raw=json.loads(archive.read('prediction.json').decode('utf-8'))
    else:
        if p.name!='prediction.json' and not p.name.endswith('_prediction.json'):
            print('Note: official filename must be prediction.json')
        raw=json.loads(p.read_text(encoding='utf-8'))
    if not isinstance(raw,list):raise ValueError('Predictions must be a JSON LIST of {id,pred} objects')
    return raw


def verify(pred,official,valid_labels):
    ids=[x['id'] for x in official];id_set=set(ids);seen=set();mapped={}
    for index,record in enumerate(pred):
        if not isinstance(record,dict) or set(record)!={'id','pred'}:
            raise ValueError(f'Record {index} must contain ONLY id and pred')
        doc_id=str(record['id'])
        if doc_id not in id_set:raise ValueError(f'Unknown official test ID: {doc_id}')
        if doc_id in seen:raise ValueError(f'Duplicate official test ID: {doc_id}')
        seen.add(doc_id)
        values=record['pred']
        if not isinstance(values,list):raise ValueError(f'{doc_id}: pred must be list')
        if not values:raise ValueError(f'{doc_id}: expected one or more statute IDs')
        if any(type(x) is not int for x in values):
            raise ValueError(f'{doc_id}: statute IDs must be INTEGER, not string/bool')
        if len(values)!=len(set(values)):raise ValueError(f'{doc_id}: duplicate statute IDs')
        invalid=set(values)-set(valid_labels)
        if invalid:raise ValueError(f'{doc_id}: unknown statute IDs {sorted(invalid)}')
        mapped[doc_id]=values
    missed=id_set-seen
    if missed:raise ValueError(f'Missing {len(missed)} IDs, including: {sorted(missed)[:5]}')
    print('PASS: official IDs, list structure, no duplicate/unknown statute IDs, nonempty output')
    return mapped


def score_dev(predmap,examples,vocabulary):
    truth=labels_matrix(examples,vocabulary)
    truth_by_id={x['id']:i for i,x in enumerate(examples)}
    index={lab:j for j,lab in enumerate(vocabulary)}
    predicted=np.zeros_like(truth)
    for k,vals in predmap.items():
        for lab in vals:predicted[truth_by_id[k],index[lab]]=1
    return multilabel_metrics(truth,predicted)


def self_test():
    records=[{'id':'x','labels':[1,7]},{'id':'y','labels':[5]}]
    raw=[{'id':'x','pred':[1,7]},{'id':'y','pred':[5]}]
    mapping=verify(raw,records,[1,5,7])
    # The official macro average also includes unused numeric IDs in 0..7.
    from phase5_official_bridge import score_dev_probability_predictions
    tiny=[{'id':'x','labels':[1,7]},{'id':'y','labels':[5]}]
    probability=np.array([[.9,.05,.9],[.01,.9,.05]],dtype=np.float32)
    met, _=score_dev_probability_predictions(tiny,[1,5,7],probability,.5)
    assert abs(met['macro_f1']-3/8)<1e-8 and met['micro_f1']==1., met
    expected_fail=[
        [{'id':'x','pred':[1,1]},{'id':'y','pred':[5]}],
        [{'id':'x','pred':[1]},{'id':'y','pred':[999]}],
        [{'id':'x','pred':[]},{'id':'y','pred':[5]}],
        [{'id':'x','pred':['1']},{'id':'y','pred':[5]}],
        [{'id':'x','pred':[True]},{'id':'y','pred':[5]}],
        [{'id':'x','pred':[1]},{'id':'x','pred':[5]}],
        [{'id':'x','pred':[1]}],
    ]
    for bad in expected_fail:
        try:verify(bad,records,[1,5,7])
        except ValueError:pass
        else:raise AssertionError('Validator failed to reject invalid prediction: '+str(bad))
    print('PASS: 7 invalid-submission rejection tests; exact official sparse-ID macro-F1 = 0.375')


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--pred',default=None,help='prediction.json or submission.zip')
    p.add_argument('--split',choices=['dev','test'],default='test')
    p.add_argument('--data-dir',default=None)
    p.add_argument('--self-test',action='store_true')
    a=p.parse_args()
    if a.self_test:self_test()
    if not a.pred:return
    train=read_examples('train',a.data_dir);dev=read_examples('dev',a.data_dir)
    label_ids=label_vocabulary(train,dev)
    official=dev if a.split=='dev' else read_examples('test',a.data_dir)
    mapping=verify(load_predictions(a.pred),official,label_ids)
    if a.split=='dev':
        # The original independent metric omitted unused statute-ID columns.
        # Use the supplied course evaluator instead, with exactly its gold schema.
        with tempfile.TemporaryDirectory() as tmp:
            truth=Path(tmp)/'dev_gold.json';prediction=Path(tmp)/'prediction.json'
            write_dev_gold(dev,truth)
            prediction.write_text(json.dumps(load_predictions(a.pred)),encoding='utf-8')
            run_official(truth,prediction)
    else:print('Test labels are private: no public test Macro-F1 can be computed.')

if __name__=='__main__':main()


Writing phase5_validate.py


In [8]:
%%writefile phase5_distill.py
#!/usr/bin/env python3
"""Phase 5 knowledge distillation: trained A3 LoRA/full TEACHER -> fewer-layer A3 STUDENT.

Teacher MUST be a trained Phase 5 multi-label classifier, not merely Phase-1 LM.
Example: python phase5_distill.py --teacher runs/a3_lora --out runs/a3_kd
"""
import argparse,time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from phase5_core import (read_examples,tokenize_examples,labels_matrix,audit_splits,
    get_p1_files,build_pretrained_backbone,StatuteModel,calibrate_thresholds,
    apply_thresholds,multilabel_metrics,save_json,set_seed)
from phase5_predict import load_trained
from phase5_train import eval_probs,choose_thresholds
from phase5_official_bridge import (valid_dev_columns, score_dev_probability_predictions,
                                    write_dev_gold, run_official)


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--teacher',required=True,help='Already TRAINED Phase 5 model directory')
    p.add_argument('--out',required=True)
    p.add_argument('--data-dir',default=None)
    p.add_argument('--layers',type=int,default=4,help='Must be fewer than the teacher layers')
    p.add_argument('--epochs',type=int,default=5)
    p.add_argument('--lr',type=float,default=6e-5)
    p.add_argument('--batch-size',type=int,default=1)
    p.add_argument('--temperature',type=float,default=2.)
    p.add_argument('--soft-weight',type=float,default=.5)
    p.add_argument('--seed',type=int,default=44)
    p.add_argument('--teacher-on-cpu',action='store_true',help='Save GPU VRAM when distilling')
    a=p.parse_args();set_seed(a.seed)
    if a.epochs<1 or a.batch_size<1 or a.temperature<=0 or not 0<=a.soft_weight<=1:
        raise ValueError('Check epochs, batch size, temperature (>0) and soft-weight (0..1)')
    out=Path(a.out);out.mkdir(parents=True,exist_ok=True)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    teach_device=torch.device('cpu') if a.teacher_on_cpu else device
    teacher,tok,tcfg,_=load_trained(a.teacher,device=teach_device)
    if tcfg['arch']!='a3':raise ValueError('This smaller-layer student uses your A3 architecture')
    if tcfg['method'] not in ('lora','full','partial','lorma'):
        raise ValueError('Need a trained Phase 5 A3 teacher')
    original_layers=int(tcfg['options']['num_layers'])
    if not 0<a.layers<original_layers:
        raise ValueError(f'Student must have 1..{original_layers-1} layers')
    train=read_examples('train',a.data_dir)
    dev=read_examples('dev',a.data_dir)
    audit_splits({'train': train, 'dev': dev}, out / 'data_audit.json')
    label_ids=tcfg['labels'];to_index={label:i for i,label in enumerate(label_ids)}
    new_labels=set(l for x in train+dev for l in x['labels'])-set(label_ids)
    if new_labels:raise ValueError('Teacher label list does not cover official train/dev labels')
    train_y=labels_matrix(train,label_ids);dev_y=labels_matrix(dev,label_ids)
    train_cache=tokenize_examples(train,tok,tcfg['length'],tcfg['chunks'])
    dev_cache=tokenize_examples(dev,tok,tcfg['length'],tcfg['chunks'])
    trainloader=DataLoader(train_cache,batch_size=a.batch_size,shuffle=True)
    devloader=DataLoader(dev_cache,batch_size=a.batch_size,shuffle=False)
    label_index={x['id']:i for i,x in enumerate(train)}
    state,student_tok,p1_cfg,prov=get_p1_files('a3',tcfg['phase1']['source'],
                                               tcfg['phase1']['weight'],out/'p1_cache')
    if len(student_tok)!=len(tok):raise ValueError('Teacher and student vocab sizes disagree')
    base,dim,options=build_pretrained_backbone('a3',state,p1_cfg,num_layers=a.layers)
    del state
    student=StatuteModel(base,'a3',dim,len(label_ids),method='kd').to(device)
    active=sum(x.numel() for x in student.parameters() if x.requires_grad)
    total=sum(x.numel() for x in student.parameters())
    config=dict(tcfg)
    config.update(method='kd',student_layers=a.layers,distilled_from=str(a.teacher),
                  teacher_method=tcfg['method'],phase1=prov,parameters_total=total,
                  parameters_trainable=active,
                  lexical_weight=0.0)  # The student uses its own logits, not the teacher TF-IDF blend.
    save_json(out/'config.json',config)
    opt=torch.optim.AdamW(student.parameters(),lr=a.lr,weight_decay=.01)
    support=train_y.sum(0)
    # Dev scorer uses every numeric ID up to the largest gold ID.
    dev_cols=valid_dev_columns(dev,label_ids)
    dev_labels=[label for label,ok in zip(label_ids,dev_cols) if ok]
    dev_gold_path=out/'dev_gold.json'
    print('Official dev label dimension:',write_dev_gold(dev,dev_gold_path))
    weight=np.clip(np.sqrt((len(train)-support+1)/(support+1)),1,12)
    posweight=torch.tensor(weight,device=device,dtype=torch.float32)
    T=a.temperature
    use_cuda=device.type=='cuda'
    use_bf16=use_cuda and torch.cuda.is_bf16_supported()
    scaler=torch.amp.GradScaler('cuda',enabled=use_cuda and not use_bf16)
    history=[];best=-1.
    for epoch in range(1,a.epochs+1):
        tic=time.time();student.train();runloss=0.
        for batch in trainloader:
            ids=batch['ids'].to(device);mask=batch['mask'].to(device)
            gold=torch.tensor(np.stack([train_y[label_index[k]] for k in batch['id']]),
                              device=device,dtype=torch.float32)
            with torch.inference_mode():
                teach_logits=teacher(batch['ids'].to(teach_device),
                                     batch['mask'].to(teach_device)).float().to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.bfloat16 if use_bf16 else torch.float16,
                                enabled=use_cuda):
                logits=student(ids,mask).float()
                hard=F.binary_cross_entropy_with_logits(logits,gold,pos_weight=posweight)
                # Use sigmoid targets because each case can have several statutes.
                soft=F.binary_cross_entropy_with_logits(logits/T,
                         torch.sigmoid(teach_logits/T))*(T*T)
                loss=(1-a.soft_weight)*hard+a.soft_weight*soft
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(student.parameters(),1.)
            scaler.step(opt);scaler.update()
            runloss+=float(loss.item())
        devprobs=eval_probs(student,devloader,device)
        threshold,cal=choose_thresholds(dev_y[:,dev_cols],devprobs[:,dev_cols],support[dev_cols],dev,dev_labels)
        met,dev_pred=score_dev_probability_predictions(
            dev,dev_labels,devprobs[:,dev_cols],threshold)
        history.append({'epoch':epoch,'loss':runloss/len(trainloader),
                        'seconds':time.time()-tic,**met})
        save_json(out/'history.json',history)
        print(f'KD epoch {epoch}: loss={runloss/len(trainloader):.4f}, macro-F1={met["macro_f1"]:.4f}')
        if met['macro_f1']>best:
            best=met['macro_f1']
            save_json(out/'dev_prediction.json',dev_pred)
            run_official(dev_gold_path,out/'dev_prediction.json')
            test_thresholds=np.full(len(label_ids),cal['global_threshold'],dtype=np.float32)
            test_thresholds[dev_cols]=threshold
            torch.save({'model':{k:v.detach().cpu() for k,v in student.state_dict().items()},
                        'config':config,'thresholds':test_thresholds.tolist(),
                        'best_dev_metrics':met},out/'best.pt')
            save_json(out/'best_metrics.json',{'epoch':epoch,**met,'calibration':cal})
    print('Student weights:',out/'best.pt','best held-out macro-F1:',best)


if __name__=='__main__':main()


Writing phase5_distill.py


## Course evaluator

This is the Phase-5 local scorer you provided, with a small dev-format converter.


In [9]:
%%writefile phase5_official_evaluator.py
#!/usr/bin/env python3
"""Local evaluator for Phase 5.

Truth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]
Prediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]

Verbatim evaluator logic supplied by the user, with no scoring changes.
"""
import argparse,json,sys
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}; max_label=-1
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or not isinstance(r['gold'],list): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        seen=set()
        for label in r['gold']:
            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f'Invalid/duplicate label in truth item {n}.')
            seen.add(label); max_label=max(max_label,label)
        out[r['id']]=r['gold']
    return out,max_label+1

def read_pred(path,nlab):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or not isinstance(r['pred'],list) or r['id'] in out: raise ValueError(f'Invalid or duplicate prediction item {n}.')
        seen=set()
        for x in r['pred']:
            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f'Invalid/duplicate label in prediction item {n}.')
            seen.add(x)
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 5 multilabel predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    print(f'Examples: {len(t)}'); print(f'Labels: {n}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}')
    micro=f1_score(yt,yp,average='micro',zero_division=0)
    subset=accuracy_score(yt,yp)
    print(f'Micro-F1: {micro:.6f}')
    print(f'Subset Accuracy: {subset:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)


Writing phase5_official_evaluator.py


In [10]:
%%writefile phase5_official_bridge.py
"""Convert Phase-5 dev files to the format expected by the course scorer."""
import subprocess
import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from phase5_official_evaluator import read_truth


def write_dev_gold(examples, path):
    """Write dev labels under the gold key expected by the evaluator."""
    import json
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = [{'id': str(row['id']), 'gold': [int(x) for x in row['labels']]} for row in examples]
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')
    # Use the organizer's original read_truth, not an approximate schema validator.
    _, label_count = read_truth(str(path))
    if label_count == 0:
        raise ValueError('Official scorer inferred zero labels; dev has no positive gold statute ID.')
    return label_count


def valid_dev_columns(examples, label_ids):
    """Return the IDs allowed by the dev evaluator."""
    highest = max((int(i) for row in examples for i in row['labels']), default=-1)
    if highest < 0:
        raise ValueError('Dev has no gold positive labels')
    return np.asarray([int(label) <= highest for label in label_ids], dtype=bool)


def score_dev_probability_predictions(examples, candidate_label_ids, probabilities, thresholds):
    """Make dev predictions and score in the official label space."""
    labels = [int(x) for x in candidate_label_ids]
    probs = np.asarray(probabilities)
    threshold = np.asarray(thresholds)
    if probs.ndim != 2 or probs.shape != (len(examples), len(labels)):
        raise ValueError('Probability matrix shape mismatch')
    if not labels:
        raise ValueError('No dev-valid label IDs')
    if threshold.ndim == 0:
        threshold = np.full(len(labels), float(threshold))
    flags = probs >= threshold[None, :]
    for row in range(len(examples)):
        if not np.any(flags[row]):
            flags[row, int(np.argmax(probs[row]))] = True
    nlab = max(int(l) for doc in examples for l in doc['labels']) + 1
    if any(label >= nlab or label < 0 for label in labels):
        raise ValueError('Dev prediction column IDs must fit official dev gold-derived label range')
    yt = np.zeros((len(examples), nlab), dtype=np.uint8)
    yp = np.zeros((len(examples), nlab), dtype=np.uint8)
    pred_records = []
    for i, doc in enumerate(examples):
        for label in doc['labels']:
            yt[i, label] = 1
        selected = sorted(labels[j] for j in np.flatnonzero(flags[i]))
        for label in selected:
            yp[i, label] = 1
        pred_records.append({'id': str(doc['id']), 'pred': selected})
    metrics = {'macro_f1': float(f1_score(yt, yp, average='macro', zero_division=0)),
               'micro_f1': float(f1_score(yt, yp, average='micro', zero_division=0)),
               'subset_accuracy': float(accuracy_score(yt, yp))}
    return metrics, pred_records


def run_official(truth_path, prediction_path):
    """Execute exact organizer script as a subprocess, failing on any format error."""
    subprocess.run([sys.executable,
                    str(Path(__file__).with_name('phase5_official_evaluator.py')),
                    '--truth', str(truth_path), '--prediction', str(prediction_path)],
                   check=True)


Writing phase5_official_bridge.py


## Method

Run: **01 A1 Full**. Train on `train.json`; compare dev Macro-F1 before submitting.


In [11]:
# Run a script and stop if it fails.
def run_py(script, *args):
    command = [sys.executable, script, *map(str, args)]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)

def data_flags():
    return ['--data-dir', DATA_DIR] if DATA_DIR else []

def source_flags():
    flags = []
    if PHASE1_SOURCE:
        flags += ['--source', PHASE1_SOURCE]
    if PHASE1_WEIGHT:
        flags += ['--weight', PHASE1_WEIGHT]
    return flags

# Quick checks before training.
run_py('phase5_validate.py', '--self-test')


Running: /usr/bin/python3 phase5_validate.py --self-test
PASS: official IDs, list structure, no duplicate/unknown statute IDs, nonempty output
PASS: 7 invalid-submission rejection tests; exact official sparse-ID macro-F1 = 0.375


In [12]:
# Train using public train data. Dev is only used for selection.
RUN_DIR = WORKDIR / 'runs/a1_full'
USE_LEXICAL_KNN = False  # Turn on to compare a train-only TF-IDF blend.
extra = ['--lexical-knn'] if USE_LEXICAL_KNN else []
run_py('phase5_train.py',
       '--arch', 'a1', '--method', 'full', '--out', str(RUN_DIR),
       '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--accum', ACCUM,
       '--length', LENGTH, '--chunks', CHUNKS, '--seed', SEED,
       *extra, *data_flags(), *source_flags())


Running: /usr/bin/python3 phase5_train.py --arch a1 --method full --out /kaggle/working/phase5_a1_full/runs/a1_full --epochs 5 --batch-size 1 --accum 4 --length 256 --chunks 3 --seed 44


/kaggle/working/phase5_a1_full/phase5_core.py:1161: UserWarning: Split overlap train__dev: {'ids': 0, 'identical_text': 81}. Review possible leakage.
  warnings.warn(f'Split overlap {pair}: {item}. Review possible leakage.')


Examples: 10181
Labels: 100
Macro-F1: 0.309325
Micro-F1: 0.401268
Subset Accuracy: 0.011590
Examples: 10181
Labels: 100
Macro-F1: 0.317192
Micro-F1: 0.409202
Subset Accuracy: 0.018073
Examples: 10181
Labels: 100
Macro-F1: 0.330040
Micro-F1: 0.422475
Subset Accuracy: 0.019350
Device: cuda Training: a1 full
Loaded Phase 5 train: 42750 documents
Loaded Phase 5 dev: 10181 documents
Official development label dimension: 100
Label vocabulary: 100 train-positive count: 158388
Parameters total 72,828,260, trainable 72,828,260
epoch 1 loss 0.342253 dev macro-F1 0.30932 dev micro-F1 0.40127
epoch 2 loss 0.293069 dev macro-F1 0.31719 dev micro-F1 0.4092
epoch 3 loss 0.259151 dev macro-F1 0.33004 dev micro-F1 0.42248
epoch 4 loss 0.222812 dev macro-F1 0.32435 dev micro-F1 0.41904
epoch 5 loss 0.187828 dev macro-F1 0.3201 dev micro-F1 0.41368
BEST held-out Macro-F1: 0.33003956828483444 checkpoint: /kaggle/working/phase5_a1_full/runs/a1_full/best.pt


## Check on dev

Scores below come from the supplied local evaluator, not a separate metric.


In [13]:
RUN_DIR = WORKDIR / 'runs/a1_full'
# Use the course scorer for the dev result.
run_py('phase5_official_evaluator.py',
       '--truth', str(RUN_DIR / 'dev_gold.json'),
       '--prediction', str(RUN_DIR / 'dev_prediction.json'))
print(json.dumps(json.loads((RUN_DIR / 'best_metrics.json').read_text()), indent=2))


Running: /usr/bin/python3 phase5_official_evaluator.py --truth /kaggle/working/phase5_a1_full/runs/a1_full/dev_gold.json --prediction /kaggle/working/phase5_a1_full/runs/a1_full/dev_prediction.json
Examples: 10181
Labels: 100
Macro-F1: 0.330040
Micro-F1: 0.422475
Subset Accuracy: 0.019350
{
  "epoch": 3,
  "macro_f1": 0.33003956828483444,
  "micro_f1": 0.4224753468325769,
  "subset_accuracy": 0.019349769177880366,
  "calibration": {
    "selected": "calibrated_per_label",
    "scores": {
      "default_0_5": {
        "macro_f1": 0.307691388753214,
        "micro_f1": 0.41395428375990423,
        "subset_accuracy": 0.02936843139180827
      },
      "calibrated_global": {
        "macro_f1": 0.30947952316520344,
        "micro_f1": 0.411055466502056,
        "subset_accuracy": 0.02023376878499165
      },
      "calibrated_per_label": {
        "macro_f1": 0.33003956828483444,
        "micro_f1": 0.4224753468325769,
        "subset_accuracy": 0.019349769177880366
      }
    },
    "gl

## Test predictions

Write `prediction.json`, package it at the ZIP root, and check IDs.


In [14]:
SUB_DIR = WORKDIR / 'submissions/a1_full'
run_py('phase5_predict.py', '--run', str(RUN_DIR), '--out', str(SUB_DIR), *data_flags(), *source_flags())
run_py('phase5_validate.py', '--pred', str(SUB_DIR / 'submission.zip'), '--split', 'test', *data_flags())
print('\nSubmission ZIP:', SUB_DIR / 'submission.zip')
print('Prediction JSON:', SUB_DIR / 'prediction.json')

Running: /usr/bin/python3 phase5_predict.py --run /kaggle/working/phase5_a1_full/runs/a1_full --out /kaggle/working/phase5_a1_full/submissions/a1_full
Loaded Phase 5 test: 13019 documents
Wrote /kaggle/working/phase5_a1_full/submissions/a1_full/prediction.json and /kaggle/working/phase5_a1_full/submissions/a1_full/submission.zip
Documents: 13019 mean predicted statutes: 5.796
Running: /usr/bin/python3 phase5_validate.py --pred /kaggle/working/phase5_a1_full/submissions/a1_full/submission.zip --split test
Loaded Phase 5 train: 42750 documents
Loaded Phase 5 dev: 10181 documents
Loaded Phase 5 test: 13019 documents
PASS: official IDs, list structure, no duplicate/unknown statute IDs, nonempty output
Test labels are private: no public test Macro-F1 can be computed.

Submission ZIP: /kaggle/working/phase5_a1_full/submissions/a1_full/submission.zip
Prediction JSON: /kaggle/working/phase5_a1_full/submissions/a1_full/prediction.json


### Run notes

- Save the `best.pt` checkpoint before the Kaggle session resets.
- Dev score is only a local estimate; Codabench has private test labels.
- Compare all six methods, not just different learning rates.
- Keep the required code and assistance disclosures in the final report.


## Save everything to Hugging Face

This uploads the whole Phase 5 experiment directory under `/kaggle/working`. It creates a **private model repo** under the username from your token. Re-running updates the same repo.

In [15]:
# Upload this experiment's /kaggle/working folder to a new private HF model repo.
# Run this after the training, dev scoring and test prediction cells.
RUN_DIR = WORKDIR / 'runs/a1_full'
SUB_DIR = WORKDIR / 'submissions/a1_full'
if not (RUN_DIR / 'best.pt').is_file():
    raise FileNotFoundError('Training checkpoint not found: ' + str(RUN_DIR / 'best.pt'))
if not (SUB_DIR / 'submission.zip').is_file():
    raise FileNotFoundError('Run the test prediction cell first: ' + str(SUB_DIR / 'submission.zip'))

# This short README goes into the repo along with the checkpoint and scripts.
metrics_file = RUN_DIR / 'best_metrics.json'
metrics = json.loads(metrics_file.read_text()) if metrics_file.exists() else {}
(WORKDIR / 'README.md').write_text(
    '# Phase 5 - A1 full fine-tuning\n\n'
    'CS781 Legal Statute Identification.\n\n'
    'Model trained using the Phase 1 checkpoint and official Phase 5 train split.\n\n'
    'Local development Macro-F1: ' + str(metrics.get('macro_f1', 'not available')) + '\n\n'
    'Files: `runs/a1_full/`, `submissions/a1_full/`, and Phase 5 scripts.\n'
)

# If this repo is already there, the next run updates its files.
hf_api.create_repo(repo_id=REPO_ID, repo_type='model', private=True,
                   exist_ok=True, token=HF_TOKEN)
print('Uploading:', WORKDIR)
hf_api.upload_folder(
    repo_id=REPO_ID,
    repo_type='model',
    folder_path=str(WORKDIR),
    token=HF_TOKEN,
    commit_message='Save Phase 5 a1_full run',
    ignore_patterns=['**/__pycache__/**', '**/*.pyc', '**/.ipynb_checkpoints/**',
                     '**/.cache/huggingface/**', '**/.git/**', '**/.env', '**/*token*secret*'],
)

# Check that the important files really reached the repo.
remote_files = set(hf_api.list_repo_files(REPO_ID, repo_type='model', token=HF_TOKEN))
for expected in ('runs/a1_full/best.pt', 'submissions/a1_full/submission.zip', 'README.md'):
    if expected not in remote_files:
        raise RuntimeError('Upload finished but file was not found on HF: ' + expected)
print('Uploaded successfully:', f'https://huggingface.co/{REPO_ID}')


Uploading: /kaggle/working/phase5_a1_full


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded successfully: https://huggingface.co/P-261110044/cs781-phase5-a1-full


**Note:** The repository is private by default. Kaggle Secrets are read at runtime and not written to the uploaded folder. Keep Internet enabled to download checkpoints and upload output files.